In [ ]:
# ============================================================
# CELL 1: LIBRARY IMPORTS
# ============================================================
# xml.etree.ElementTree — build and serialize XML tree structures
# xml.dom.minidom      — pretty-print XML output
# pandas               — data loading, manipulation, and DataFrame operations
# html                 — HTML escape/unescape text safely for web output
# ast                  — safely parse Python literal strings (e.g., lists stored as text)
# re                   — regular expressions for pattern matching in text

from xml.etree.ElementTree import Element, SubElement, tostring
from xml.dom import minidom
import pandas as pd
import html
import ast
import re
import ast
import html
import pandas as pd


In [ ]:
# ============================================================
# CELL 2: HELPER — split_citation_precision(text)
# ============================================================
# Splits a combined explanation string (from the 'Explanation_Citation' column)
# into two parts separated at the second newline:
#   - 'general'  → the first two lines (high-level label/summary)
#   - 'details'  → everything after the second newline (detailed explanation)
# The result is used to create two new DataFrame columns:
#   'Citation_Precision_General' and 'Citation_Precision_Details'
# These feed into the provenance detail panel rendered in the HTML output.

def split_citation_precision(text):
    text = str(text) if pd.notnull(text) else ''
    parts = text.split('\n', 2)  # Split into at most 3 parts
    if len(parts) >= 3:
        general = parts[0] + '\n' + parts[1]
        details = parts[2]
    elif len(parts) == 2:
        general = parts[0] + '\n' + parts[1]
        details = ''
    else:
        general = parts[0] if parts else ''
        details = ''
    return pd.Series([general, details])

In [ ]:
# ============================================================
# CELL 3: create_provenance_map(df)
# ============================================================
# Builds a dictionary mapping each provenance ID to its structured metadata.
# Each entry contains:
#   - generated_sentence        : The original AI-generated sentence
#   - sources                   : List of source papers (title, ID, sentence, category)
#   - citation_accuracy         : High-level accuracy label (e.g., 'Accurate', 'Partial')
#   - explanation               : Merged general + detailed accuracy explanation
#   - transformation_explanation: Description of how the sentence was edited
# This map is consumed by generate_html_with_accordion() to populate the
# clickable detail side panels for each (Prov.N) marker in the HTML.

def create_provenance_map(df):
    """
    Parses details from a DataFrame to create the structured provenance map.

    This function creates a dictionary where each value is another dictionary
    containing structured data (generated_sentence, sources, explanation, etc.),
    which is the format expected by the `generate_html_with_accordion` function.
    """
    provenance_map = {}

    # Define essential columns to check for non-null values
    subset_cols = ['Id_Provenance', 'Source_Details', 'Generated_Sentence']
    unique_prov_df = df.dropna(subset=subset_cols).drop_duplicates(subset=['Id_Provenance'])

    for _, row in unique_prov_df.iterrows():
        prov_id = row['Id_Provenance']

        # Initialize the dictionary to hold all structured details
        details_dict = {
            "generated_sentence": row.get('Generated_Sentence', 'N/A'),
            "sources": [],
            "citation_accuracy": row.get('Citation_Precision', 'N/A'),
            "explanation": "N/A",
            "transformation_explanation": row.get('Explanation_Transformation', 'N/A')
        }

        # --- Parse the Source Details ---
        source_details_str = row['Source_Details']
        sources_list = []
        try:
            source_data = ast.literal_eval(source_details_str)
            if isinstance(source_data, list):
                for source_item in source_data:
                    sources_list.append({
                        "paper_title": source_item.get('Paper_Title', 'N/A'),
                        "provenance_id": source_item.get('Provenance', 'N/A'),
                        "source_sentence": source_item.get('Sentence', source_item.get('Text_Original', 'N/A')),
                        "category": source_item.get('Category', 'N/A')
                    })
        except (ValueError, SyntaxError):
            print(f"Warning: Could not parse Source_Details for {prov_id}")

        details_dict['sources'] = sources_list

        # --- Combine General and Detailed Explanations ---
        explanation_parts = []
        exp_general = row.get('Citation_Precision_General')
        if pd.notnull(exp_general) and str(exp_general).strip():
            explanation_parts.append(str(exp_general))

        exp_details = row.get('Citation_Precision_Details')
        if pd.notnull(exp_details) and str(exp_details).strip():
            explanation_parts.append(str(exp_details))

        if explanation_parts:
            details_dict['explanation'] = "\n\n".join(explanation_parts)

        # --- Assign the completed dictionary to the main map ---
        provenance_map[prov_id] = details_dict

    return provenance_map

In [ ]:
# ============================================================
# CELL 4: XML TEXT GENERATION
# ============================================================
# _create_xml_from_column(df, column_name) — INTERNAL HELPER
#   Groups the DataFrame by Title and Paragraph, deduplicates sentences
#   within each paragraph, and serializes the content into an XML-like
#   string using <title> and <paragraph> tags.
#
# generate_xml_texts(df) — PUBLIC FUNCTION
#   Calls the helper twice to produce two parallel text representations:
#     1. xml_generated — uses 'Generated_Sentence' (raw AI output, pre-editing)
#     2. xml_final     — uses 'Final_Sentence'     (post-editing, human-reviewed)
#   Both share the same structural skeleton (titles/paragraphs).
#   xml_generated is used to extract citations; xml_final is rendered in the HTML.

# It generates an xml of the complete related work section.
def _create_xml_from_column(df, column_name):
    """
    Internal helper function to generate an XML-like string
    from a specific sentence column.
    """
    xml_elements = []
    for title, title_group in df.groupby('Title', sort=False):
        xml_elements.append(f'<title>{html.escape(title)}</title>')
        for paragraph, paragraph_group in title_group.groupby('Paragraph', sort=False):
            # Use the specified column_name to get sentences
            sentences = paragraph_group[column_name].tolist()

            # Remove duplicates while preserving order
            seen = set()
            unique_sentences = [s for s in sentences if not (s in seen or seen.add(s))]

            paragraph_text = ' '.join(str(s) for s in unique_sentences)
            xml_elements.append(f'<paragraph>{html.escape(paragraph_text)}</paragraph>')

    return '\n'.join(xml_elements)

def generate_xml_texts(df):
    """
    Generates two XML-like strings from the DataFrame: one using the
    'Generated_Sentence' column and another using the 'Final_Sentence' column.

    Args:
        df (pd.DataFrame): DataFrame with 'Title', 'Paragraph',
                           'Generated_Sentence', and 'Final_Sentence' columns.

    Returns:
        tuple: A tuple containing two strings:
               (xml_from_generated_sentence, xml_from_final_sentence)
    """
    # --- Step 1: Prepare the DataFrame (do this only once) ---
    # Ensure 'Paragraph' is an integer for correct sorting
    df['Paragraph'] = df['Paragraph'].astype(int)
    # Sort once to ensure order for both outputs
    df_sorted = df.sort_values(by=['Title', 'Paragraph'])

    # --- Step 2: Generate each XML text by calling the helper function ---
    xml_generated = _create_xml_from_column(df_sorted, 'Generated_Sentence')
    xml_final = _create_xml_from_column(df_sorted, 'Final_Sentence')

    # --- Step 3: Return both results ---
    return xml_generated, xml_final


In [ ]:
# ============================================================
# CELL 5: HTML GENERATOR — generate_html_with_accordion()
# ============================================================
# Produces a complete, self-contained Bootstrap 5 HTML page that presents
# the AI-generated Related Work section as an interactive accordion UI.
#
# Key features of the generated page:
#   - Each paper/topic title becomes a collapsible accordion panel.
#   - Citation markers like [1], [2] are turned into clickable spans;
#     clicking opens a side panel showing the reference's title, year, source.
#   - Provenance markers like (Prov.1) are turned into clickable spans;
#     clicking opens a side panel with the generated sentence, citation accuracy
#     type, detailed explanation, transformation edits, and source paper info.
#   - A final 'Bibliography' accordion item lists all references with <hr> separators.
#   - JavaScript handlers manage show/hide of the right-side info panel and
#     highlight the currently selected provenance link.
#
# Args:
#   xml_content     : XML-like string from generate_xml_texts() (Final_Sentence version)
#   provenance_map  : dict from create_provenance_map()
#   df_bibliography : DataFrame with Citation_Partial, Full_Citation, Title_Reference,
#                     Year, and Source columns


def generate_html_with_accordion(xml_content, provenance_map, df_bibliography):
  """
  Generates a complete HTML page with a Bootstrap accordion and logic
  for handling non-overlapping secondary popovers with close buttons.

  Args:
      xml_content (str): The string content containing the related work sections.
      provenance_map (dict): A dictionary mapping provenance IDs to their details.
                             (Assumes the new, structured format).
      df_bibliography (pd.DataFrame): A DataFrame containing bibliography info.
                                      Required columns: 'Citation_Partial', 'Full_Citation',
                                      'Title_Reference', 'Year', 'Source'.
  """
  accordion_items_html = ""
  content_chunks = xml_content.split('<title>')[1:]

  # Prepare a sorted list of citations for safe replacement (longest first)
  sorted_citations = []
  if df_bibliography is not None and not df_bibliography.empty and 'Citation_Partial' in df_bibliography.columns:
      temp_citations = []
      for _, row in df_bibliography.iterrows():
          temp_citations.append((
              row['Citation_Partial'],
              row.get('Title_Reference', 'N/A'),
              row.get('Year', 'N/A'),
              row.get('Source', 'N/A')
          ))
      sorted_citations = sorted(temp_citations, key=lambda x: len(x[0]), reverse=True)


  for i, chunk in enumerate(content_chunks):
      parts = chunk.split('</title>')
      title_text = parts[0]
      body_content = parts[1].strip()
      collapse_id = f"flush-collapse-{i}"
      header_id = f"flush-header-{i}"
      title_display = html.unescape(title_text)
      body_display = html.unescape(body_content).replace('\n', '<br>')

      # 1. Add click-triggered popovers for citations first
      if sorted_citations:
          for marker, title, year, source in sorted_citations:
              popover_content = (
                  f"<strong>Title: {html.escape(str(title))}</strong><br>"
                  f"<strong>Year:</strong> {html.escape(str(year))}<br>"
                  f"<strong>Source:</strong> {html.escape(str(source))}"
              )
              escaped_content = html.escape(popover_content, quote=True)
              replacement_html = (
                  f'<span class="citation-marker" '
                  f'data-title="Citation Details" '
                  f'data-content="{escaped_content}">'
                  f'{html.escape(marker)}</span>'
              )
              body_display = body_display.replace(html.escape(marker), replacement_html)

      # 2. Add click-triggered popovers for provenance links
      def create_popover_link(match):
          prov_id = match.group(0)
          details = provenance_map.get(prov_id)

          if not details:
              popover_content = "Details not found."
          else:
              def info_item(label, value):
                  escaped_value = html.escape(str(value)).replace('\n', '<br>')
                  return f'<div class="info-item"><strong>{label}:</strong> {escaped_value}</div>'

              html_parts = []

              # 1. Section: Generated Sentence
              if details.get('generated_sentence'):
                  html_parts.append('<h6 class="info-section-title">Generated Sentence</h6>')
                  html_parts.append(f'<p class="info-generated-sentence">{html.escape(details["generated_sentence"])}</p>')

              # 2. Section: Citation Accuracy
              if details.get('citation_accuracy') or (details.get('explanation') and details.get('explanation') != 'N/A'):
                  html_parts.append('<h6 class="info-section-title">Citation Accuracy</h6>')
                  html_parts.append('<div class="info-paper-block">')
                  if details.get('citation_accuracy'):
                      html_parts.append(info_item("Type", details['citation_accuracy']))
                  if details.get('explanation') and details.get('explanation') != 'N/A':
                      html_parts.append(info_item("Explanation", details['explanation']))
                  html_parts.append('</div>')

              # 3. Section: Transformation Edits
              if details.get('transformation_explanation') and details.get('transformation_explanation') != 'N/A':
                   html_parts.append('<h6 class="info-section-title">Transformation Edits</h6>')
                   html_parts.append(f"""<div class="info-paper-block">
                                           <p>{html.escape(details['transformation_explanation']).replace('\\n', '<br>')}</p>
                                        </div>""")

              # 4. Section: Paper sources
              if details.get('sources'):
                  html_parts.append('<h6 class="info-section-title">Paper</h6>')
                  for source in details['sources']:
                      html_parts.append('<div class="info-paper-block">')
                      if source.get('paper_title'):
                          html_parts.append(f'<p class="info-paper-title">{html.escape(source["paper_title"])}</p>')
                      if source.get('provenance_id'):
                          html_parts.append(info_item("Provenance", source['provenance_id']))
                      if source.get('source_sentence'):
                          html_parts.append(info_item("Source Sentence", source['source_sentence']))
                      if source.get('category'):
                          html_parts.append(info_item("Category", source['category']))
                      html_parts.append('</div>')

              popover_content = "".join(html_parts)

          escaped_content = html.escape(popover_content, quote=True)
          return (
              f'<span class="provenance-link" '
              f'data-title="Provenance Details - {prov_id}" '
              f'data-content="{escaped_content}">{prov_id}</span>'
          )
      body_with_popovers = re.sub(r'\(Prov\.\d+\)', create_popover_link, body_display)

      item_html = f"""
      <div class="accordion-item">
        <h2 class="accordion-header" id="{header_id}">
          <button class="accordion-button collapsed" type="button" data-bs-toggle="collapse" data-bs-target="#{collapse_id}" aria-expanded="false" aria-controls="{collapse_id}">
            {title_display}
            <span class="toggle-icon me-2"></span>
          </button>
        </h2>
        <div id="{collapse_id}" class="accordion-collapse collapse" aria-labelledby="{header_id}" data-bs-parent="#accordionContainer">
          <div class="accordion-body">
            {body_with_popovers.replace('<paragraph>', '<p>').replace('</paragraph>', '</p>')}
          </div>
        </div>
      </div>
      """
      accordion_items_html += item_html

  if df_bibliography is not None and not df_bibliography.empty and 'Full_Citation' in df_bibliography.columns:
      num_citations = len(df_bibliography)
      citation_text = "citation" if num_citations == 1 else "citations"
      bibliography_title = f"Bibliography ({num_citations} {citation_text})"

      # --- START: MODIFIED BLOCK ---
      bibliography_body_content = ""
      citations_list = df_bibliography['Full_Citation'].tolist()
      num_citations_total = len(citations_list)

      for i, citation in enumerate(citations_list):
          # Add the citation paragraph
          bibliography_body_content += f"<p>{html.escape(citation)}</p>"
          # Add a horizontal rule, but not after the very last citation
          if i < num_citations_total - 1:
              bibliography_body_content += "<hr>"
      # --- END: MODIFIED BLOCK ---

      bibliography_accordion_item = f"""
      <div class="accordion-item">
        <h2 class="accordion-header" id="flush-header-bibliography">
          <button class="accordion-button collapsed" type="button" data-bs-toggle="collapse" data-bs-target="#flush-collapse-bibliography" aria-expanded="false" aria-controls="flush-collapse-bibliography">
            {bibliography_title}
            <span class="toggle-icon me-2"></span>
          </button>
        </h2>
        <div id="flush-collapse-bibliography" class="accordion-collapse collapse" aria-labelledby="flush-header-bibliography" data-bs-parent="#accordionContainer">
          <div class="accordion-body">
            {bibliography_body_content}
          </div>
        </div>
      </div>
      """
      accordion_items_html += bibliography_accordion_item


  html_template = f"""
<!doctype html>
<html lang="en">
  <head>
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.2/dist/css/bootstrap.min.css" rel="stylesheet">
    <title>Generated Related Work</title>
    <link rel="preconnect" href="https://fonts.googleapis.com">
    <link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
    <link href="https://fonts.googleapis.com/css2?family=Inter:ital,opsz,wght@0,14..32,100..900;1,14..32,100..900&display=swap" rel="stylesheet">

<style>
  :root {{
    --black: #000;
    --white: #fff;
    --purple: #8a7ed9;
    --indigo: #4946a6;
    --navy: #282c40;
    --steel-blue: #6f92bf;
    --blue: #8f92bf;
    --grey-dark: #333333;
    --grey: #dcd8f4;
    --grey-light: #c5d3e5;
    --grey-pale: #ebebeb;
    --grey-border: #e6e6e6;
    --grey-text: #272727;
    --grey-text-dark: #292929;
  }}

  body {{
    background-color: var(--white);
    color: var(--navy);
    font-family: 'Inter', sans-serif;
    font-size:15px;
    font-weight:400;
  }}

  /* --- CSS CHANGE 1: WIDEN THE PAGE CONTAINER --- */
  .page-container {{
    max-width: 1400px; /* Increased from 1200px */
    margin: auto;
  }}

  .accordion-container {{
    width: 100%;
  }}

  .accordion {{
    background-color: transparent;
    border: none;
    --bs-accordion-border-width: 0;
  }}

  .accordion-header{{
    background: var(--grey-pale);
  }}

  .accordion-item {{
    background-color: var(--grey-pale);
    border: 1px solid var(--grey-border);
    border-radius: 0.5rem;
    margin-bottom: 1rem;
    box-shadow: 0 0.125rem 0.25rem rgba(0, 0, 0, 0.075);
    transition: box-shadow 0.2s ease-in-out;
  }}

  .accordion-item:hover {{
    box-shadow: 0 0.25rem 0.5rem rgba(0, 0, 0, 0.1);
    background:var(--grey-pale);
  }}

  .accordion-button {{
    background:var(--grey-pale);
    font-weight: 500;
    border-radius: 0.5rem;
    padding: 1.25rem 1.5rem;
    position: relative;
    display: flex;
    align-items: center;
    justify-content: space-between;
    width: 100%;
    text-align: left;
  }}

  .accordion-button:not(.collapsed) {{
    background-color: var(--grey-pale);
    color:var(--black);
  }}

  .accordion-body {{
    text-align: justify;
    padding: 1.5rem;
    font-size: 1rem;
    line-height: 1.6;
  }}

  .accordion-body p {{
    margin-bottom: 1.25rem;
  }}

  .accordion-button::after {{
    display: none; /* remove Bootstrap arrow */
  }}

  .toggle-icon{{
    background:var(--grey);
    text-align: center;
    padding:4px 10px;
    color:var(--black);
    border-radius: 4px;
    margin-left: auto; /* Pushes the icon to the far right */
  }}

  .toggle-icon::before {{
    content: "+";
    font-size: 1.25rem;
    font-weight: bold;
    color: var(--black);
  }}

  .accordion-button:not(.collapsed) .toggle-icon::before {{
    content: "−";
  }}

  .accordion-button:focus{{
    border-color:var(--grey-pale);
    box-shadow: none;
  }}

  .provenance-link,
  .citation-marker {{
    color: var(--indigo);
    font-weight: 700;
    font-family: 'Inter', sans-serif;
    text-decoration: none;
    cursor: pointer;
    border-bottom: 1px dashed var(--indigo);
    padding: 0 4px;
    border-radius: 4px;
  }}

  .provenance-link:hover,
  .citation-marker:hover {{
    color: var(--indigo);
    text-decoration: underline;
  }}

  .provenance-link:active,
  .citation-marker:active {{
    color: var(--purple);
    text-decoration: underline;
  }}

  .provenance-link.selected {{
    background-color: var(--steel-blue);
    color: var(--grey-text-dark);
    text-decoration: none;
  }}

  #infoPanel {{
    height: 80vh;
    overflow-y: auto;
    background: #fff;
    border: 1px solid var(--grey-border);
    border-radius: 10px;
    box-shadow: 0 0.125rem 0.25rem rgba(0, 0, 0, 0.075);
    display: none;
    font-size: 14px;
  }}

  #infoPanel.active {{
    display: block;
  }}

  #infoPanel h5 {{
    font-size: 16px;
    display: flex;
    justify-content: space-between;
    align-items: center;
  }}

  #infoTitle{{
    background:var(--grey-light);
    font-size:16px;
    font-weight: 500;
    padding: 1rem;
    position: sticky;
    top: 0;
    z-index: 10;
  }}

  .close-panel {{
    background: none;
    border: none;
    font-size: 1.5rem;
    line-height: 1;
    cursor: pointer;
    color: #6c757d;
  }}

  .close-panel:hover {{
    color: #000;
  }}

  #infoContent{{
    padding: 0.5rem 1rem 1rem 1rem;
  }}

  #infoContent h6.info-section-title {{
      font-weight: 700;
      font-size: 1rem;
      color: var(--navy);
      margin-top: 1.5rem;
      margin-bottom: 0.75rem;
      padding-bottom: 0.25rem;
      border-bottom: 1px solid var(--grey-border);
  }}

  #infoContent h6.info-section-title:first-child {{
      margin-top: 0.5rem;
  }}

  .info-generated-sentence {{
      background-color: var(--grey-pale);
      border-left: 3px solid var(--steel-blue);
      padding: 0.75rem;
      margin-bottom: 1rem;
      font-style: italic;
  }}

  .info-paper-block {{
      border: 1px solid var(--grey-border);
      border-radius: 0.25rem;
      padding: 0.75rem;
      margin-bottom: 1rem;
  }}

  .info-paper-block p:last-child,
  .info-paper-block .info-item:last-child {{
    margin-bottom: 0;
  }}

  p.info-paper-title {{
      font-weight: 600;
      margin-bottom: 0.5rem;
  }}

  .info-item {{
      margin-bottom: 0.5rem;
      line-height: 1.5;
      text-align: justify;
  }}

  /* --- CSS CHANGE 2: MAKE LABELS INLINE --- */
  .info-item strong {{
      color: var(--navy);
      margin-right: 0.5em; /* Add space between label and text */
      /* REMOVED fixed width and display properties to allow natural flow */
  }}

  .header-bar {{
    display: flex;
    justify-content: space-between;
    align-items: center;
    margin-bottom: 1rem;
    border-bottom: 3px solid var(--grey-border);
    padding-bottom: 0.5rem;
  }}

  .btn-about {{
    font-family: 'Inter', sans-serif;
    font-weight: 700;
    border: none;
    border-radius: 25px;
    padding: 10px 15px;
    background-color: var(--indigo);
    color: white;
    transition: background-color 0.2s ease-in-out;
  }}

  .btn-about:hover {{
    background-color: var(--purple);
    color: white;
  }}

  .btn-about.selected {{
    background-color: var(--steel-blue);
    color: var(--grey-text-dark);
  }}
</style>
  </head>
  <body>
    <div class="container page-container accordion-container my-5">
      <div class="header-bar">
         <h1 class="mb-0">AI-generated Related Work Section</h1>
         <button class="btn btn-outline-secondary btn-sm btn-about">About</button>
      </div>
      <div class="row">
        <div class="col-md-7">
           <div class="accordion" id="accordionContainer">
             {accordion_items_html}
           </div>
         </div>
         <div class="col-md-5">
           <div id="infoPanel">
             <h5 id="infoTitle">Details <button class="close-panel" onclick="hideInfoPanel()">&times;</button></h5>
             <div id="infoContent"></div>
           </div>
         </div>
      </div>
    </div>

    <script src="https://cdn.jsdelivr.net/npm/bootstrap@5.3.2/dist/js/bootstrap.bundle.min.js" integrity="sha384-C6RzsynM9kWDrMNeT87bh95OGNyZPhcTNXj1NW7RuBCsyN/o0jlpcV8Qyq46cDfL" crossorigin="anonymous"></script>

    <script>
      function showInfoPanel(title, content) {{
        document.getElementById('infoTitle').innerHTML = `${{title}} <button class="close-panel" onclick="hideInfoPanel()">&times;</button>`;
        document.getElementById('infoContent').innerHTML = content;
        document.getElementById('infoPanel').classList.add('active');
      }}

      function hideInfoPanel() {{
        document.getElementById('infoPanel').classList.remove('active');
        document.querySelectorAll('.provenance-link.selected').forEach(l => l.classList.remove('selected'));
      }}

      document.querySelectorAll('.citation-marker, .provenance-link').forEach(el => {{
        el.addEventListener('click', (event) => {{
          event.stopPropagation();
          const title = el.getAttribute('data-title') || 'Details';
          const content = el.getAttribute('data-content');
          showInfoPanel(title, content);
        }});
      }});

      document.querySelectorAll('.provenance-link').forEach(link => {{
        link.addEventListener('click', () => {{
          document.querySelectorAll('.provenance-link').forEach(l => l.classList.remove('selected'));
          link.classList.add('selected');
        }});
      }});
    </script>
  </body>
</html>
"""
  return html_template


In [ ]:
# ============================================================
# CELL 6: DATA LOADING & PREPROCESSING
# ============================================================
# 1. Load the main Excel dataset containing sentences, provenance IDs,
#    citation accuracy labels, and transformation explanations.
# 2. Append a period to each Final_Sentence to ensure proper punctuation.
# 3. Strip internal annotation tags (e.g., '_b1') from Final_Sentence.
# 4. Normalize provenance IDs to clean strings.
# 5. Split the 'Explanation_Citation' column into two columns using
#    split_citation_precision(): 'Citation_Precision_General' and 'Citation_Precision_Details'.
# 6. Build the provenance_map dictionary for the HTML detail panels.
# 7. Generate both XML text versions (original AI output + final edited version).

file_path = "Jodi_Draft_CitationAccuracyv01_Sentence.xlsx"
# file_path = "Jodi_Draft_CitationAccuracyv01_NoSentence.xlsx"
# file_path = "Related_Work_GPT5_Jodi_Sentences.xlsx"

df = pd.read_excel(file_path)
print("File loaded successfully.")
df["Final_Sentence"] = df["Final_Sentence"] + "."
df["Final_Sentence"] = df["Final_Sentence"].str.replace(r"_b\d+", "", regex=True) # new
df['Id_Provenance'] = df['Id_Provenance'].astype(str).str.strip()
df[['Citation_Precision_General', 'Citation_Precision_Details']] = df['Explanation_Citation'].apply(split_citation_precision)
print("DataFrame processed.")
provenance_map = create_provenance_map(df)
print(f"Provenance map created with {len(provenance_map)} entries.")
text_original, text_formatted = generate_xml_texts(df)



File loaded successfully.
DataFrame processed.
Provenance map created with 50 entries.


In [ ]:
# ============================================================
# CELL 7: LOAD & DEDUPLICATE REFERENCES
# ============================================================
# Loads the bibliography Excel file containing all cited papers.
# Steps:
#   1. Assign an occurrence counter per Title_Reference to identify duplicates.
#   2. Keep only the first occurrence of each referenced title (Id_Title == 1).
#   3. Apply title-case formatting to Authors, Title_Reference, and Source
#      for consistent display in the bibliography section of the HTML.

# Reading the References.
reference = pd.read_excel("Test_Jodi_Referencesv01_Inter.xlsx")
reference['Id_Title'] = reference.groupby('Title_Reference').cumcount() + 1
reference= reference[reference['Id_Title'] == 1]
reference['Authors'] = reference['Authors'].str.title()
reference['Title_Reference'] = reference['Title_Reference'].str.title()
reference['Source'] = reference['Source'].str.title()
reference

,Paper_Id,Title,Citation_Id,Authors,Year,Title_Reference,Source,PaperId,Frequency,Citation_Standard,Index,Id_Title
0,paper7,why do some retracted articles continue to get...,@b36,A Mott; C Fairhurst; D Torgerson,2019,Assessing The Impact Of Retraction On The Cita...,Journal Of Health Services Research & Policy,paper7,4,[1],1,1
4,paper7,why do some retracted articles continue to get...,@b57,E M Suelzer; J Deal; K L Hanus; B Ruggeri; R S...,2019,Assessment Of Citations Of The Retracted Artic...,Jama Network Open,paper7,4,[2],2,1
8,paper7,why do some retracted articles continue to get...,@b2,J Bar-Ilan; G Halevi,2017,Post Retraction Citations In Context: A Case S...,Scientometrics,paper7,4,[3],3,1
12,paper7,why do some retracted articles continue to get...,@b49,X Shuai; J Rollins; I Moulinier; T Custis; M E...,2017,A Multidimensional Investigation Of The Effect...,Journal Of The Association For Information Sci...,paper7,3,[4],4,1
15,paper7,why do some retracted articles continue to get...,@b45,M Schmidt,2018,An Analysis Of The Validity Of Retraction Anno...,Journal Of The Association For Information Sci...,paper7,3,[5],5,1
...,...,...,...,...,...,...,...,...,...,...,...,...
231,paper5,continued post-retraction citation of a fraudu...,@b49,G Halevi,2019,Why Articles In Arts And Humanities Are Being ...,Publishing Research Quarterly,paper5,1,[335],335,1
232,paper3,what difference might retractions make? an est...,@b12,J P A Ioannidis,2005,Why Most Published Research Findings Are False,Plos Med. Public Library Of Science,paper3,1,[337],337,1
233,paper7,why do some retracted articles continue to get...,@citation338,Marion Schmidt,0,Why Do Some Retracted Articles Continue To Get...,NaN,NaN,1,[338],338,1
234,paper2,addressing unreliability propagation in scient...,@citation339,Heng Zheng; Yuanxi Fu; M Sarol; Ishita Sarraf;...,0,Addressing Unreliability Propagation In Scient...,NaN,NaN,1,[339],339,1


In [ ]:
# ============================================================
# CELL 8: get_citation(text) — CITATION PARSER
# ============================================================
# Scans the XML text and extracts two types of citation markers:
#   1. Standard citations:  [123]       → type 'Citation_Standard', Id = 123
#   2. Provenance tuples:   (123, 456)  → type 'Provenance', Id = 123 (first number)
# Returns a list of dicts, each with keys: Type, Citation_Standard, Id.
# Used in Cell 9 to build the citation index table.

def get_citation(text):
    """
    Parses a text string to find and extract standard citations and provenance data.
    """
    square_bracket_pattern = r'\[(\d+)\]'  # Matches [123]
    tuple_pattern = r'\((\d+),\s*(\d+)\)'  # Matches (123, 456)
    matches = []

    # Find square bracketed numbers in the single text string
    square_numbers = re.findall(square_bracket_pattern, text)
    for num in square_numbers:
        matches.append({'Type': 'Citation_Standard', 'Citation_Standard': f'[{num}]', 'Id': int(num)})

    # Find tuple patterns in the single text string
    tuples = re.findall(tuple_pattern, text)
    for num1, num2 in tuples:
        matches.append({'Type': 'Provenance', 'Citation_Standard': f'({num1}, {num2})', 'Id': int(num1)})

    return matches



In [ ]:
# ============================================================
# CELL 9: BUILD CITATION INDEX TABLE (df_citation)
# ============================================================
# Parses the original AI-generated XML text to collect all citation references.
# Steps:
#   1. Call get_citation() to extract all [N] and (N, M) markers.
#   2. Normalize provenance-style citations to [N] format for consistent merging.
#   3. Deduplicate entries by Citation_Standard + Id.
#   4. Sort: standard citations first (by type order), then by numeric Id.
#   5. Assign a sequential 'Id_Final' (1, 2, 3, ...) which becomes the
#      new display number used in both the in-text citations and the bibliography.

results = get_citation(text_original)
df_citation = pd.DataFrame(results)

# Convert Provenance citation format (num1, num2) to [num1]
df_citation.loc[df_citation['Type'] == 'Provenance', 'Citation_Standard'] = df_citation.loc[
    df_citation['Type'] == 'Provenance', 'Id'
].apply(lambda x: f'[{x}]')

df_citation = df_citation.drop_duplicates(subset=['Citation_Standard', 'Id'])

# Step 1: Define the desired order for Type
type_order = {'Provenance': 0, 'Citation_Standard': 1}

# Step 2: Sort by Type (custom order) and then by Id
df_citation = df_citation.sort_values(by=['Type', 'Id'], key=lambda col: col.map(type_order) if col.name == 'Type' else col).reset_index(drop=True)

# Step 3: Assign increasing Id_Final
df_citation['Id_Final'] = range(1, len(df_citation) + 1)
df_citation

,Type,Citation_Standard,Id,Id_Final
0,Citation_Standard,[2],2,1
1,Citation_Standard,[3],3,2
2,Citation_Standard,[7],7,3
3,Citation_Standard,[18],18,4
4,Citation_Standard,[20],20,5
5,Citation_Standard,[22],22,6
6,Citation_Standard,[33],33,7
7,Citation_Standard,[34],34,8
8,Citation_Standard,[39],39,9
9,Citation_Standard,[42],42,10


In [ ]:
# ============================================================
# CELL 10: BUILD FINAL BIBLIOGRAPHY (bibliography DataFrame)
# ============================================================
# Merges the references table with the citation index to obtain only
# the references actually cited in the generated text.
# Steps:
#   1. Inner join on 'Citation_Standard' to filter to cited references.
#   2. Fill missing Authors/Source with placeholder strings.
#   3. Build 'Full_Citation': a human-readable string formatted as
#      '[N] Authors, Title, Year, Source'
#   4. Replace the original [N] number in Full_Citation with the new Id_Final number.
#   5. Create 'Citation_Partial' = '[Id_Final]' — the short in-text label.
#   6. Sort by Id_Final for ordered display in the HTML bibliography section.

# Getting final bibliography.
bibliography = pd.merge(reference, df_citation, on='Citation_Standard')
bibliography['Authors'] = bibliography['Authors'].fillna('Author')
bibliography['Source'] = bibliography['Source'].fillna('Source')
bibliography['Full_Citation'] =bibliography['Citation_Standard']+' '+ bibliography['Authors'] + ', ' + bibliography['Title_Reference'] + ', ' +bibliography['Year'].astype(str) + ', ' + bibliography['Source']
bibliography['Full_Citation'] = bibliography.apply(
    lambda row: re.sub(r'\[\d+\]', f"[{row['Id_Final']}]", row['Full_Citation']),
    axis=1
)
bibliography['Citation_Partial'] = '[' + bibliography['Id_Final'].astype(str) + ']'

bibliography = bibliography.sort_values(by='Id_Final', ascending=True)


bibliography

,Paper_Id,Title,Citation_Id,Authors,Year,Title_Reference,Source,PaperId,Frequency,Citation_Standard,Index,Id_Title,Type,Id,Id_Final,Full_Citation,Citation_Partial
0,paper7,why do some retracted articles continue to get...,@b57,E M Suelzer; J Deal; K L Hanus; B Ruggeri; R S...,2019,Assessment Of Citations Of The Retracted Artic...,Jama Network Open,paper7,4,[2],2,1,Citation_Standard,2,1,[1] E M Suelzer; J Deal; K L Hanus; B Ruggeri;...,[1]
1,paper7,why do some retracted articles continue to get...,@b2,J Bar-Ilan; G Halevi,2017,Post Retraction Citations In Context: A Case S...,Scientometrics,paper7,4,[3],3,1,Citation_Standard,3,2,"[2] J Bar-Ilan; G Halevi, Post Retraction Cita...",[2]
2,paper2,addressing unreliability propagation in scient...,@b16,Henry Small,1982,Citation Context Analysis,Source,paper2,3,[7],7,1,Citation_Standard,7,3,"[3] Henry Small, Citation Context Analysis, 19...",[3]
3,paper6,critical citations in knowledge construction a...,@b64,S Teufel; A Siddharthan; D Tidhar,2006,Automatic Classification Of Citation Function,Source,paper6,2,[18],18,1,Citation_Standard,18,4,"[4] S Teufel; A Siddharthan; D Tidhar, Automat...",[4]
4,paper6,critical citations in knowledge construction a...,@b25,M Hernández-Alvarez; J M Gomez Soriano; P Mart...,2017,"Citation Function, Polarity And Influence Clas...",Natural Language Engineering,paper6,2,[20],20,1,Citation_Standard,20,5,[5] M Hernández-Alvarez; J M Gomez Soriano; P ...,[5]
5,paper2,addressing unreliability propagation in scient...,@b4,Daryl E Chubin; Soumyo D Moitra,1975,Content Analysis Of References: Adjunct Or Alt...,Social Studies Of Science,paper2,2,[22],22,1,Citation_Standard,22,6,"[6] Daryl E Chubin; Soumyo D Moitra, Content A...",[6]
6,paper4,continued use of retracted papers: temporal tr...,@b57,M Valenzuela; V Ha; O Etzioni,2015,Identifying Meaningful Citations,Source,paper4,2,[33],33,1,Citation_Standard,33,7,"[7] M Valenzuela; V Ha; O Etzioni, Identifying...",[7]
7,paper6,critical citations in knowledge construction a...,@b31,R Jha; A.-A Jbara; V Qazvinian; D R Radev,2017,Nlp-Driven Citation Analysis For Scientometrics,Natural Language Engineering,paper6,2,[34],34,1,Citation_Standard,34,8,"[8] R Jha; A.-A Jbara; V Qazvinian; D R Radev,...",[8]
8,paper6,critical citations in knowledge construction a...,@b44,M J Moravcsik; P Murugesan,1975,Some Results On The Function And Quality Of Ci...,Social Studies Of Science,paper6,2,[39],39,1,Citation_Standard,39,9,"[9] M J Moravcsik; P Murugesan, Some Results O...",[9]
9,paper4,continued use of retracted papers: temporal tr...,@b44,M P Pfeifer; G L Snodgrass,1990,"The Continued Use Of Retracted, Invalid Scient...",Jama,paper4,2,[42],42,1,Citation_Standard,42,10,"[10] M P Pfeifer; G L Snodgrass, The Continued...",[10]


In [ ]:
# ============================================================
# CELL 11: UPDATE IN-TEXT CITATIONS IN FORMATTED TEXT
# ============================================================
# The final (edited) text was built using original citation numbers like [42], [114].
# This cell replaces each original citation with its renumbered equivalent
# (from 'Citation_Partial', e.g., [10], [19]) so that in-text citations
# match the sequential numbering in the bibliography section of the HTML.
# The updated text is saved back into 'text_formatted'.

### Updating the references on the formatted text (the final text).

# Create a copy of the text to modify
text_with_partial_citations = text_formatted

# Iterate over each row of the bibliography DataFrame
for index, row in bibliography.iterrows():
    # Get the text to find (e.g., '[1]')
    text_to_find = row['Citation_Standard']

    # Get the text to replace it with from the 'Citation_Partial' column
    replacement_text = row['Citation_Partial'] # <-- This is the corrected line

    # Perform the replacement
    text_with_partial_citations = text_with_partial_citations.replace(
        text_to_find, replacement_text
    )

text_formatted=text_with_partial_citations


In [ ]:
# ============================================================
# CELL 12: GENERATE & SAVE HTML OUTPUT
# ============================================================
# Calls generate_html_with_accordion() with:
#   - text_formatted  : the final related work text with updated citation numbers
#   - provenance_map  : structured metadata for each (Prov.N) marker
#   - bibliography    : the final reference list with full citations
# The resulting HTML string is written to an .html file.
# Opening the file in a browser displays the full interactive Related Work UI.

# file_name="Test_Jodi_GPT5_v01.html" # Changed version number for the new output file
# file_name="Test_Jodi_NoSentence_Prov_v01.html" # Changed version number for the new output file
file_name="Test_Jodi_Sentence_Prov_v01.html" # Changed version number for the new output file


my_html_page = generate_html_with_accordion(text_formatted, provenance_map,bibliography)

with open(file_name, "w", encoding="utf-8") as f:
  f.write(my_html_page)
  print(f"\nSuccessfully created '{file_name}'. Open it in your browser to see the result.")



Successfully created 'Test_Jodi_Sentence_Prov_v01.html'. Open it in your browser to see the result.


str